In [0]:
%sql

create schema if not exists databrickslearning.silver;


In [0]:
bronze_table='databrickslearning.bronze.patients_raw'
silver_table='databrickslearning.silver.dim_patient'
checkpoint_path="abfss://data@databrickslearningsa.dfs.core.windows.net/silver/dim_patient/checkpoint"

from pyspark.sql.functions import sha2, col, current_timestamp, monotonically_increasing_id, concat_ws

df_patient_bronze = (
    spark.readStream.table(bronze_table)
)

df_patient_clean = (
    df_patient_bronze
      .dropDuplicates(["patient_id"])
      .withColumn("patient_first_last_name_masked", sha2(concat_ws('|',col("first_name"),col("last_name")), 256))
      .withColumn("load_timestamp", current_timestamp())
)

In [0]:
from delta.tables import DeltaTable

def  merge_dim_patient (batch_df, batch_id):
    if not spark.catalog.tableExists(silver_table):
        batch_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
        return
    
#load delta table by name and upsert
    dim_patient = DeltaTable.forName(spark,silver_table)

    (dim_patient.alias("t")
        .merge(

            batch_df.alias("s"),
            "t.patient_id = s.patient_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


In [0]:
(
    df_patient_clean.writeStream
        .foreachBatch(merge_dim_patient)
        .outputMode("update")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .start()
)